# 02 - Load Bronze
Reads each raw Parquet file from `Files/landing/` and writes it as a Delta table in the
**`bronze`** schema (`bronze.<table>`).

> **Coexistence narrative:** in production the **SAP** tables (`sap_fi_cost`, `sap_mm_po`,
> `sap_supplier`) arrive via **SAP BDC Connect / mirroring** (zero-copy from S/4HANA on Azure
> via RISE), while the **non-SAP** tables (schedule, engineering change, project/WBS master)
> arrive via **OneLake shortcuts** (data stays in Primavera / PC&E, connected in place). For
> the synthetic demo they are all local Delta, but the origin_system column keeps the story literal.


In [ ]:
LANDING = 'Files/landing'
tables = ['dim_project', 'dim_wbs', 'fact_schedule_activity', 'sap_fi_cost', 'sap_mm_po', 'sap_supplier', 'fact_engineering_change', 'ext_disruption_signal']

spark.sql('CREATE SCHEMA IF NOT EXISTS bronze')
for t in tables:
    df = spark.read.parquet(f'{LANDING}/{t}.parquet')
    (df.write.format('delta').mode('overwrite')
        .option('overwriteSchema', 'true').saveAsTable(f'bronze.{t}'))
    print(f'bronze.{t:26s} {df.count():>8,} rows')
print('Bronze load complete.')
